### Imports & Hyperparameters

In [2]:
import copy, torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from trl import SFTTrainer, SFTConfig

In [ ]:
N_DOMAINS = 3
EXPERTS_PER_DOMAIN = 4

### Model definition

In [4]:
base_config = AutoConfig.from_pretrained("ibm-granite/granite-3.0-1b-a400m-base")
print(base_config)

GraniteMoeConfig {
  "architectures": [
    "GraniteMoeForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "attention_multiplier": 0.015625,
  "bos_token_id": 0,
  "embedding_multiplier": 12.0,
  "eos_token_id": 0,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 512,
  "logits_scaling": 6.0,
  "max_position_embeddings": 4096,
  "model_type": "granitemoe",
  "num_attention_heads": 16,
  "num_experts_per_tok": 8,
  "num_hidden_layers": 24,
  "num_key_value_heads": 8,
  "num_local_experts": 32,
  "output_router_logits": false,
  "pad_token_id": 0,
  "residual_multiplier": 0.22,
  "rms_norm_eps": 1e-06,
  "rope_parameters": {
    "rope_theta": 10000,
    "rope_type": "default"
  },
  "router_aux_loss_coef": 0.001,
  "tie_word_embeddings": true,
  "transformers_version": "5.22.2",
  "use_cache": true,
  "vocab_size": 49152
}



In [ ]:
BASE = "ibm-granite/granite-3.0-1b-a400m-base"

tok  = AutoTokenizer.from_pretrained(BASE)
base_model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.bfloat16)
n_old = base_model.config.num_local_experts            # 32 for this model

Loading weights: 100%|██████████| 219/219 [00:00<00:00, 245.57it/s]


In [ ]:
config_final = AutoConfig.from_pretrained("ibm-granite/granite-3.0-1b-a400m-base")
config_final.num_local_experts = n_old + EXPERTS_PER_DOMAIN * N_DOMAINS
final_model = AutoModelForCausalLM.from_config(config_final, dtype=torch.bfloat16)
old_sd, final_sd = base_model.state_dict(), final_model.state_dict()
for k, v in final_sd.items():
    o = old_sd[k]
    if o.shape == v.shape:
        final_sd[k] = o
    else:
        assert o.shape[0] < v.shape[0], f"{k}: {o.shape} !< {v.shape}"
        final_sd[k][:o.shape[0]] = o
final_model.load_state_dict(final_sd)

### Expert extension

In [ ]:
config_new = AutoConfig.from_pretrained("ibm-granite/granite-3.0-1b-a400m-base")
config_new.num_local_experts = n_old + EXPERTS_PER_DOMAIN
model = AutoModelForCausalLM.from_config(config_new, dtype=torch.bfloat16)

old_sd, new_sd = base_model.state_dict(), model.state_dict()
src = torch.randint(0, n_old, (EXPERTS_PER_DOMAIN,))          # each new expert starts as a copy of an old one
for k, v in new_sd.items():
    o = old_sd[k]
    if o.shape == v.shape:
        new_sd[k] = o
    else:                                         # router / expert tensors: dim 0 = expert
        extra = o[src].clone().float()
        extra += 0.01 * o.float().std() * torch.randn_like(extra)   # break symmetry
        new_sd[k] = torch.cat([o, extra.to(o.dtype)], 0)
model.load_state_dict(new_sd)

In [ ]:
del base_model

### Model freezing

In [ ]:
for p in model.parameters():
    p.requires_grad_(False)

def only_new(grad):
    grad = grad.clone(); grad[:n_old] = 0; return grad

for layer in model.model.layers:
    moe = layer.block_sparse_moe
    for p in (moe.experts.gate_up_proj, moe.experts.down_proj, moe.router.weight):
        p.requires_grad_(True)
        p.register_hook(only_new)

### Domain Datasets

In [36]:
import re

WP   = re.compile(r"^\s*\[\s*WP\s*\]", re.IGNORECASE)   # filter: must start with [ WP ]
TAGS = re.compile(r"^(\s*\[[^\]]*\]\s*)+")               # strip: all leading [ .. ] tags
N, N_TEST = 40_000, 500

def prep(name, p, c, fmt=lambda x: x, cfmt=lambda x: x, keep=None, split="train"):
    ds = load_dataset(name, split=split)
    if keep is not None:
        ds = ds.filter(keep)
    ds = ds.shuffle(seed=0).select(range(min(N + N_TEST, len(ds))))
    ds = ds.map(lambda ex: {"prompt": fmt(ex[p]) + "\n\n", "completion": cfmt(ex[c])},
                remove_columns=ds.column_names)
    return ds.train_test_split(test_size=N_TEST, shuffle=False)   # already shuffled above

clean = lambda s: s.replace(" <newline> ", "\n").replace("<newline>", "\n").strip()

domains = {
    "math":     prep("meta-math/MetaMathQA", "query", "response",
                     fmt=lambda q: f"Question: {q}\nAnswer:"),
    "code":     prep("ise-uiuc/Magicoder-OSS-Instruct-75K", "problem", "solution"),
    "creative": prep("euclaise/writingprompts", "prompt", "story",
                     keep=lambda ex: WP.match(ex["prompt"]) is not None,
                     fmt=lambda s: clean(TAGS.sub("", s)),
                     cfmt=clean),
}

In [35]:
bad = [p for p in domains["creative"]["train"]["prompt"] if p.lstrip().startswith("[")]
assert not bad, f"{len(bad)} tagged prompts left, e.g. {bad[:3]}"

In [32]:
import textwrap

def show(ds, name, n=5, width=300):
    print(f"\n{'=' * 30} {name.upper()} ({len(ds):,} rows) {'=' * 30}")
    for i, ex in enumerate(ds.shuffle(seed=42).select(range(n))):
        p = textwrap.shorten(ex["prompt"], width, placeholder=" …")
        c = textwrap.shorten(ex["completion"], width, placeholder=" …")
        print(f"\n--- [{i}] ---\nPROMPT:     {p!r}\nCOMPLETION: {c!r}")

for name, ds in domains.items():
    show(ds, name)


============================== MATH (40,000 rows) ==============================

--- [0] ---
PROMPT:     'Question: Define $E(a,b,c) = a \\times b^c$. What positive value of $r$ is the solution to the equation $E(r,r,3) = 625$? Answer:'
COMPLETION: 'We have $E(r,r,3) = r \\times r^3 = r^4$. Setting this equal to 625, we have $r^4 = 625$. Taking the fourth root of both sides, we get $r = \\sqrt[4]{625}$. Simplifying the right side, we have $r = \\sqrt[4]{5^4 \\times 5} = \\sqrt[4]{5^5} = \\boxed{5}$. The answer is: 5'

--- [1] ---
PROMPT:     'Question: Don throws 3 darts. One is a bullseye worth x points. One completely missed the target, so received no points. The third was worth half the points of the bullseye. What is the final score from these 3 throws? If we know the answer to the above question is 75, what is the value of unknown …'
COMPLETION: 'One dart is a bullseye worth x points. One dart completely missed the target, so received no points. The third dart was worth half the

In [31]:
# assert not any(s.lstrip().startswith("[") for s in domains["creative"]["prompt"])
# show(domains["creative"], "creative")

for prompt in domains["creative"]["prompt"]:
    if prompt.lstrip().startswith("["):
        print(f"ERROR: prompt starts with a tag: {prompt!r}")

### TLR/SFT Training

In [ ]:
# ToDo:
# - train one model per domain, then combine them into a single model with 4 new experts per domain
# - make one large for loop and execute on the cluster

In [ ]:
ds = load_dataset("json", data_files="train.jsonl", split="train")  # {"text": ...} or {"prompt","completion"}

args = SFTConfig(
    output_dir="granite-moe-40e",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    weight_decay=0.0,                 # important, see below
    num_train_epochs=1,
    max_length=2048,
    packing=True,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},  # needed because the embeddings are frozen
    router_aux_loss_coef=0.01,        # load balancing so the new experts actually get tokens
    logging_steps=10,
)
trainer = SFTTrainer(model=model, args=args, train_dataset=ds, processing_class=tok)
trainer.train()
trainer.save_model()